# Index Table Preparation

In [2]:
import geopandas as gpd
import pandas as pd
import requests
import json
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit


# config
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

Pandas notes for me: 

gdf.loc[ROWS, COLUMNS] = "select rows by this value, and columns by this value"

new field:
gdf["new_field"] = (
    gdf["field_a"].astype(str)
    + "_"
    + gdf["field_b"].astype(str)
    + "_"
    + gdf["field_c"].astype(str)
)

### Load the Shapefile and give it a true unique ID before doing anything else to it. 

In [3]:
# this is the path to the original feature service, downloaded as a shapefile from here: 
# https://www.arcgis.com/home/item.html?id=f260e4065a384e7ebee05b23362f380f#overview
# Layer 36112 - https://gis5.uspatial.umn.edu/arcgis/rest/services/mhapo/MHAPO_dynamic/MapServer/0
original_index_path = "data/historic_airphoto_index/shapefiles/umn_airphoto_index_original.shp"

gdf_original = gpd.read_file(original_index_path)

print(gdf_original.crs)
# we're keeping this in web mercator for now. 

EPSG:3857


In [4]:
# make a unique identifier

gdf_original["source_id"] = range(1, len(gdf_original) + 1)

assert gdf_original["source_id"].is_unique
assert gdf_original["source_id"].notna().all()

In [ ]:
gdf_original

In [7]:
gdf_original.to_file(
    "data/historic_airphoto_index/umn_index_original_with_source_id.gpkg",
    driver="GPKG"
)

### Cleaning the original table

In [8]:
gdf_clean = gdf_original.copy()

#preview the columns
gdf_clean.columns

Index(['flight_cod', 'url_link', 'photo_id', 'lat', 'long_', 'year_',
       'objectid_1', 'in_fid', 'near_fid', 'near_dist', 'objectid_2',
       'statefp10', 'countyfp10', 'countyns10', 'geoid10', 'name10',
       'namelsad10', 'lsad10', 'classfp10', 'mtfcc10', 'csafp10', 'cbsafp10',
       'metdivfp10', 'funcstat10', 'aland10', 'awater10', 'intptlat10',
       'intptlon10', 'year_conve', 'decade', 'decade_dat', 'decade_str',
       'created_us', 'created_da', 'last_edite', 'last_edi_1', 'geometry',
       'source_id'],
      dtype='str')

#### Cleaning the URLs
And looking for true duplicates...

In [9]:
# clean URL spaces

gdf_clean["url_link"] = gdf_clean["url_link"].str.strip()

In [40]:
exists = 'https://geo.lib.umn.edu/Olmsted/1940/WB-3A-115.jpg' in gdf_clean['url_link'].values

exists

False

In [ ]:
# identify duplicate URLs

duplicates = gdf_clean[gdf_clean["url_link"].duplicated(keep=False)]
duplicates = duplicates.sort_values(by=['flight_cod','url_link'])
duplicates

,flight_cod,url_link,photo_id,lat,long_,year_,objectid_1,in_fid,near_fid,near_dist,objectid_2,statefp10,countyfp10,countyns10,geoid10,name10,namelsad10,lsad10,classfp10,mtfcc10,csafp10,cbsafp10,metdivfp10,funcstat10,aland10,awater10,intptlat10,intptlon10,year_conve,decade,decade_dat,decade_str,created_us,created_da,last_edite,last_edi_1,geometry,source_id
110838,Hennepin,https://geo.lib.umn.edu/Hennepin_County/y1937/WN-10-867.jpg,WN-10-867,45.163490,-93.371491,1937.0,112117,112117,47,0.0,47,27,053,00659472,27053,Hennepin,Hennepin County,06,H1,G4020,378,33460,None,A,1.433794e+09,137155436.0,+45.0060644,-093.4751846,1937-01-01,1930,NaT,1930,None,NaT,BORCHERT,2025-12-15,POINT (-10394066.859 5647296.475),110839
110840,Hennepin,https://geo.lib.umn.edu/Hennepin_County/y1937/WN-10-867.jpg,WN-10-867,45.185798,-93.371902,1937.0,112119,112119,76,0.0,76,27,003,00659447,27003,Anoka,Anoka County,06,H1,G4020,378,33460,None,A,1.095590e+09,59831077.0,+45.2741098,-093.2427232,1937-01-01,1930,NaT,1930,None,NaT,BORCHERT,2025-12-15,POINT (-10394112.585 5650819.159),110841
115716,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-3A-101.jpg,WB-3A-101,43.860098,-92.079152,1940.0,0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.000000e+00,0.0,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10250204.313 5443817.49),115717
115717,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-3A-101.jpg,WB-3A-103,43.860376,-92.121252,1940.0,0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.000000e+00,0.0,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10254890.84 5443860.455),115718
115718,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-3A-101.jpg,WB-3A-105,43.860655,-92.163352,1940.0,0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.000000e+00,0.0,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10259577.367 5443903.419),115719
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48032,koo,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/projects/koo/y1988/koo_35_052.jpg,koo_35_052,47.926300,-93.339203,1988.0,49248,49220,53,0.0,53,27,071,00659481,27071,Koochiching,Koochiching County,06,H1,G4020,NaN,NaN,None,A,8.039504e+09,129375868.0,+48.2454895,-093.7828417,1988-01-01,1980,NaT,1980,None,NaT,BORCHERT,2026-07-31,POINT (-10390472.538 6094602.527),48033
19098,wp,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/projects/wp/y1963/wp_1dd_235.jpg,wp_1dd_235,45.196300,-93.662903,1963.0,20314,20286,68,0.0,68,27,171,00659530,27171,Wright,Wright County,06,H1,G4020,378,33460,None,A,1.713165e+09,136781631.0,+45.1750914,-093.9663965,1963-01-01,1960,NaT,1960,None,NaT,BORCHERT,2026-07-31,POINT (-10426506.646 5652478.103),19099
19099,wp,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/projects/wp/y1963/wp_1dd_235.jpg,wp_1dd_235,45.195499,-93.642502,1963.0,20315,20287,47,0.0,47,27,053,00659472,27053,Hennepin,Hennepin County,06,H1,G4020,378,33460,None,A,1.433794e+09,137155436.0,+45.0060644,-093.4751846,1963-01-01,1960,NaT,1960,None,NaT,BORCHERT,2026-07-31,POINT (-10424235.617 5652351.553),19100
14664,wp,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/projects/wp/y1963/wp_2dd_209.jpg,wp_2dd_209,44.994202,-94.099800,1963.0,15418,14790,68,0.0,68,27,171,00659530,27171,Wright,Wright County,06,H1,G4020,378,33460,None,A,1.713165e+09,136781631.0,+45.1750914,-093.9663965,1963-01-01,1960,NaT,1960,None,NaT,BORCHERT,2026-07-31,POINT (-10475141.831 5620608.709),14665


In [ ]:
# identify duplicate URLs with unique photo_ids to be fixed

fake_duplicates = gdf_clean[
    gdf_clean["url_link"].duplicated(keep=False)
    & ~gdf_clean["photo_id"].duplicated(keep=False)]
fake_duplicates = fake_duplicates.sort_values(by=['flight_cod','url_link'])
fake_duplicates

,flight_cod,url_link,photo_id,lat,long_,year_,objectid_1,in_fid,near_fid,near_dist,objectid_2,statefp10,countyfp10,countyns10,geoid10,name10,namelsad10,lsad10,classfp10,mtfcc10,csafp10,cbsafp10,metdivfp10,funcstat10,aland10,awater10,intptlat10,intptlon10,year_conve,decade,decade_dat,decade_str,created_us,created_da,last_edite,last_edi_1,geometry,source_id
115716,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-3A-101.jpg,WB-3A-101,43.860098,-92.079152,1940.0,0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,0.0,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10250204.313 5443817.49),115717
115717,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-3A-101.jpg,WB-3A-103,43.860376,-92.121252,1940.0,0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,0.0,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10254890.84 5443860.455),115718
115718,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-3A-101.jpg,WB-3A-105,43.860655,-92.163352,1940.0,0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,0.0,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10259577.367 5443903.419),115719
115719,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-3A-101.jpg,WB-3A-107,43.860933,-92.205451,1940.0,0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,0.0,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10264263.894 5443946.384),115720
115720,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-3A-101.jpg,WB-3A-109,43.861211,-92.247551,1940.0,0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,0.0,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10268950.42 5443989.348),115721
115721,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-3A-101.jpg,WB-3A-111,43.861489,-92.289651,1940.0,0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,0.0,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10273636.947 5444032.314),115722
115722,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-3A-101.jpg,WB-3A-113,43.861768,-92.331751,1940.0,0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,0.0,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10278323.473 5444075.278),115723
115723,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-3A-101.jpg,WB-3A-115,43.862046,-92.373851,1940.0,0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,0.0,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10283010 5444118.244),115724
115724,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-3A-101.jpg,WB-3A-117,43.862324,-92.415950,1940.0,0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,0.0,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10287696.528 5444161.211),115725
115725,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-3A-101.jpg,WB-3A-119,43.862603,-92.458050,1940.0,0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,0.0,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10292383.054 5444204.176),115726


In [ ]:
# fix olmsted county URLs

bad_url = "https://geo.lib.umn.edu/Olmsted/1940/WB-3A-101.jpg"

mask = gdf_clean["url_link"] == bad_url

gdf_clean.loc[mask, "url_link"] = (
    "https://geo.lib.umn.edu/Olmsted/1940/"
    + gdf_clean.loc[mask, "photo_id"]
    + ".jpg"
)

In [15]:
check = gdf_clean[gdf_clean["flight_cod"] == "Olmsted"]

check

,flight_cod,url_link,photo_id,lat,long_,year_,objectid_1,in_fid,near_fid,near_dist,objectid_2,statefp10,countyfp10,countyns10,geoid10,name10,namelsad10,lsad10,classfp10,mtfcc10,csafp10,cbsafp10,metdivfp10,funcstat10,aland10,awater10,intptlat10,intptlon10,year_conve,decade,decade_dat,decade_str,created_us,created_da,last_edite,last_edi_1,geometry,source_id
115588,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-1A-003.jpg,WB-1A-003,44.181032,-92.303203,1940.0,0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,0.0,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10275145.556 5493500.176),115589
115589,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-1A-005.jpg,WB-1A-005,44.181175,-92.347837,1940.0,0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,0.0,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10280114.246 5493522.412),115590
115590,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-1A-007.jpg,WB-1A-007,44.181319,-92.392472,1940.0,0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,0.0,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10285082.936 5493544.649),115591
115591,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-1A-009.jpg,WB-1A-009,44.181462,-92.437106,1940.0,0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,0.0,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10290051.626 5493566.885),115592
115592,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-1A-011.jpg,WB-1A-011,44.181605,-92.481741,1940.0,0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,0.0,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10295020.316 5493589.121),115593
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115744,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-4A-041.jpg,WB-4A-041,43.948613,-92.515435,1940.0,0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,0.0,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10298771.078 5457493.316),115745
115745,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-4A-043.jpg,WB-4A-043,43.948634,-92.558712,1940.0,0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,0.0,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10303588.744 5457496.692),115746
115746,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-4A-045.jpg,WB-4A-045,43.948656,-92.601990,1940.0,0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,0.0,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10308406.41 5457500.067),115747
115747,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-4A-047.jpg,WB-4A-047,43.948678,-92.645268,1940.0,0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.0,0.0,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10313224.077 5457503.444),115748


In [16]:
# identify duplicate URLs again

duplicates = gdf_clean[gdf_clean["url_link"].duplicated(keep=False)]
duplicates = duplicates.sort_values(by=['flight_cod','url_link'])
duplicates

,flight_cod,url_link,photo_id,lat,long_,year_,objectid_1,in_fid,near_fid,near_dist,objectid_2,statefp10,countyfp10,countyns10,geoid10,name10,namelsad10,lsad10,classfp10,mtfcc10,csafp10,cbsafp10,metdivfp10,funcstat10,aland10,awater10,intptlat10,intptlon10,year_conve,decade,decade_dat,decade_str,created_us,created_da,last_edite,last_edi_1,geometry,source_id
110838,Hennepin,https://geo.lib.umn.edu/Hennepin_County/y1937/WN-10-867.jpg,WN-10-867,45.163490,-93.371491,1937.0,112117,112117,47,0.0,47,27,053,00659472,27053,Hennepin,Hennepin County,06,H1,G4020,378,33460,None,A,1.433794e+09,137155436.0,+45.0060644,-093.4751846,1937-01-01,1930,NaT,1930,None,NaT,BORCHERT,2025-12-15,POINT (-10394066.859 5647296.475),110839
110840,Hennepin,https://geo.lib.umn.edu/Hennepin_County/y1937/WN-10-867.jpg,WN-10-867,45.185798,-93.371902,1937.0,112119,112119,76,0.0,76,27,003,00659447,27003,Anoka,Anoka County,06,H1,G4020,378,33460,None,A,1.095590e+09,59831077.0,+45.2741098,-093.2427232,1937-01-01,1930,NaT,1930,None,NaT,BORCHERT,2025-12-15,POINT (-10394112.585 5650819.159),110841
55760,ait,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/projects/ait/y1975/ait_6_144.jpg,ait_6_144,47.007702,-93.069901,1975.0,56976,56961,66,0.0,66,27,001,00663198,27001,Aitkin,Aitkin County,06,H1,G4020,NaN,NaN,None,A,4.718084e+09,449700337.0,+46.6024464,-093.4197600,1975-01-01,1970,NaT,1970,None,NaT,BORCHERT,2026-07-31,POINT (-10360493.932 5943331.309),55761
57982,ait,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/projects/ait/y1975/ait_6_144.jpg,ait_6_144,46.524502,-93.137703,1975.0,59198,59198,66,0.0,66,27,001,00663198,27001,Aitkin,Aitkin County,06,H1,G4020,NaN,NaN,None,A,4.718084e+09,449700337.0,+46.6024464,-093.4197600,1975-01-01,1970,NaT,1970,None,NaT,BORCHERT,2026-07-31,POINT (-10368041.66 5864803.111),57983
56600,ait,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/projects/ait/y1982/ait_1_100.jpg,ait_1_100,46.952801,-93.761101,1982.0,57816,57816,66,0.0,66,27,001,00663198,27001,Aitkin,Aitkin County,06,H1,G4020,NaN,NaN,None,A,4.718084e+09,449700337.0,+46.6024464,-093.4197600,1982-01-01,1980,NaT,1980,None,NaT,BORCHERT,2026-07-31,POINT (-10437437.997 5934373.362),56601
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48032,koo,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/projects/koo/y1988/koo_35_052.jpg,koo_35_052,47.926300,-93.339203,1988.0,49248,49220,53,0.0,53,27,071,00659481,27071,Koochiching,Koochiching County,06,H1,G4020,NaN,NaN,None,A,8.039504e+09,129375868.0,+48.2454895,-093.7828417,1988-01-01,1980,NaT,1980,None,NaT,BORCHERT,2026-07-31,POINT (-10390472.538 6094602.527),48033
19098,wp,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/projects/wp/y1963/wp_1dd_235.jpg,wp_1dd_235,45.196300,-93.662903,1963.0,20314,20286,68,0.0,68,27,171,00659530,27171,Wright,Wright County,06,H1,G4020,378,33460,None,A,1.713165e+09,136781631.0,+45.1750914,-093.9663965,1963-01-01,1960,NaT,1960,None,NaT,BORCHERT,2026-07-31,POINT (-10426506.646 5652478.103),19099
19099,wp,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/projects/wp/y1963/wp_1dd_235.jpg,wp_1dd_235,45.195499,-93.642502,1963.0,20315,20287,47,0.0,47,27,053,00659472,27053,Hennepin,Hennepin County,06,H1,G4020,378,33460,None,A,1.433794e+09,137155436.0,+45.0060644,-093.4751846,1963-01-01,1960,NaT,1960,None,NaT,BORCHERT,2026-07-31,POINT (-10424235.617 5652351.553),19100
14664,wp,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/projects/wp/y1963/wp_2dd_209.jpg,wp_2dd_209,44.994202,-94.099800,1963.0,15418,14790,68,0.0,68,27,171,00659530,27171,Wright,Wright County,06,H1,G4020,378,33460,None,A,1.713165e+09,136781631.0,+45.1750914,-093.9663965,1963-01-01,1960,NaT,1960,None,NaT,BORCHERT,2026-07-31,POINT (-10475141.831 5620608.709),14665


In [17]:
# drop duplicates arbitrarily (the only thing that varies is the geometry and we're not using it.)
gdf_clean = gdf_clean.drop_duplicates(
    subset=["url_link"],
    keep="first"
).copy()

In [25]:
# identify duplicate URLs again
# there should be none. 

duplicates = gdf_clean[gdf_clean["photo_id"].duplicated(keep=False)]
duplicates = duplicates.sort_values(by=['flight_cod','photo_id'])
duplicates

,flight_cod,url_link,photo_id,lat,long_,year_,objectid_1,in_fid,near_fid,near_dist,objectid_2,statefp10,countyfp10,countyns10,geoid10,name10,namelsad10,lsad10,classfp10,mtfcc10,csafp10,cbsafp10,metdivfp10,funcstat10,aland10,awater10,intptlat10,intptlon10,year_conve,decade,decade_dat,decade_str,created_us,created_da,last_edite,last_edi_1,geometry,source_id
99281,Hennepin,https://geo.lib.umn.edu/Dakota_County/y1940/WK-4A-145.jpg,WN-4A-145,45.048962,-93.418001,1940,100560,100560,47,0.000000,47,27,053,00659472,27053,Hennepin,Hennepin County,06,H1,G4020,378,33460,None,A,1.433794e+09,1.371554e+08,+45.0060644,-093.4751846,1940-01-01,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10399244.302 5629232.847),99282
112241,Hennepin,https://geo.lib.umn.edu/Hennepin_County/y1940/WN-4A-145.jpg,WN-4A-145,45.049315,-93.418311,1940,0,0,0,0.000000,0,NaN,NaN,NaN,NaN,Hennepin,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.000000e+00,0.000000e+00,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10399278.789 5629288.499),112242
97701,Washington,https://geo.lib.umn.edu/washington/y1936/ccz-1-10.jpg,ccz-1-10,45.222364,-93.001213,1936,98980,98980,34,0.000000,34,27,163,00659526,27163,Washington,Washington County,06,H1,G4020,378,33460,None,A,9.952855e+08,9.961890e+07,+45.0379290,-092.8901170,1936-01-01,1930,NaT,1930,None,NaT,BORCHERT,2025-12-15,POINT (-10352847.674 5656596.342),97702
97891,Washington,https://geo.lib.umn.edu/washington/y1938/ccz-1-10.jpg,ccz-1-10,44.898016,-92.761724,1938,99170,99170,34,1169.694738,34,27,163,00659526,27163,Washington,Washington County,06,H1,G4020,378,33460,None,A,9.952855e+08,9.961890e+07,+45.0379290,-092.8901170,1938-01-01,1930,NaT,1930,None,NaT,BORCHERT,2025-12-15,POINT (-10326187.881 5605480.456),97892
97703,Washington,https://geo.lib.umn.edu/washington/y1936/ccz-1-12.jpg,ccz-1-12,45.249043,-93.000414,1936,98982,98982,34,0.000000,34,27,163,00659526,27163,Washington,Washington County,06,H1,G4020,378,33460,None,A,9.952855e+08,9.961890e+07,+45.0379290,-092.8901170,1936-01-01,1930,NaT,1930,None,NaT,BORCHERT,2025-12-15,POINT (-10352758.73 5660813.79),97704
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66235,stl,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/projects/stl/y1981/stl_028_105.jpg,stl_028_105,47.663601,-92.936997,1981,67451,67213,32,0.000000,32,27,137,00662850,27137,St. Louis,St. Louis County,06,H1,G4020,NaN,20260,None,A,1.618069e+10,1.585336e+09,+47.5838524,-092.4636448,1981-01-01,1980,NaT,1980,None,NaT,BORCHERT,2026-07-31,POINT (-10345699.126 6051071.368),66236
61878,stl,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/projects/stl/y1989/stl_028_106.jpg,stl_028_106,46.838799,-92.226402,1989,63094,63094,32,0.000000,32,27,137,00662850,27137,St. Louis,St. Louis County,06,H1,G4020,NaN,20260,None,A,1.618069e+10,1.585336e+09,+47.5838524,-092.4636448,1989-01-01,1980,NaT,1980,None,NaT,BORCHERT,2026-07-31,POINT (-10266596.142 5915801.464),61879
66236,stl,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/projects/stl/y1981/stl_028_106.jpg,stl_028_106,47.662399,-92.955299,1981,67452,67214,32,0.000000,32,27,137,00662850,27137,St. Louis,St. Louis County,06,H1,G4020,NaN,20260,None,A,1.618069e+10,1.585336e+09,+47.5838524,-092.4636448,1981-01-01,1980,NaT,1980,None,NaT,BORCHERT,2026-07-31,POINT (-10347736.596 6050872.758),66237
61879,stl,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/projects/stl/y1989/stl_028_107.jpg,stl_028_107,46.824902,-92.225800,1989,63095,63095,32,0.000000,32,27,137,00662850,27137,St. Louis,St. Louis County,06,H1,G4020,NaN,20260,None,A,1.618069e+10,1.585336e+09,+47.5838524,-092.4636448,1989-01-01,1980,NaT,1980,None,NaT,BORCHERT,2026-07-31,POINT (-10266529.05 5913540.242),61880


In [ ]:
print("Unique URLs:", gdf_clean["url_link"].is_unique)
print("Remaining records:", len(gdf_clean))

Unique URLs: True
Unique photo_ids: False
Remaining records: 115577


### Clean up other values

In [20]:
# convert year field to an integer rather than a float
# handles missing values
gdf_clean["year_"] = gdf_clean["year_"].astype("Int64")

In [21]:
# populate missing counties from the flight code field
gdf_clean["name10"] = gdf_clean["name10"].fillna(
    gdf_clean["flight_cod"]
)

In [22]:
# check for nulls
print(gdf_clean["name10"].isna().any())

print(gdf_clean["year_"].isna().any())

print(gdf_clean["photo_id"].isna().any())

print(gdf_clean["url_link"].isna().any())

False
False
False
False


In [23]:
# check it out
gdf_clean

,flight_cod,url_link,photo_id,lat,long_,year_,objectid_1,in_fid,near_fid,near_dist,objectid_2,statefp10,countyfp10,countyns10,geoid10,name10,namelsad10,lsad10,classfp10,mtfcc10,csafp10,cbsafp10,metdivfp10,funcstat10,aland10,awater10,intptlat10,intptlon10,year_conve,decade,decade_dat,decade_str,created_us,created_da,last_edite,last_edi_1,geometry,source_id
0,bxq,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxq/y1939/bxq07185.jpg,bxq07185,46.964901,-94.303902,1939,1,1,85,0.0,85,27,021,00659456,27021,Cass,Cass County,06,H1,G4020,NaN,14660,None,A,5.235770e+09,1.016740e+09,+46.9514271,-094.3337731,1939-01-01,1930,NaT,1930,None,NaT,BORCHERT,2026-07-31,POINT (-10497862.317 5936346.903),1
1,bxq,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxq/y1939/bxq07187.jpg,bxq07187,46.965599,-94.346802,1939,2,2,85,0.0,85,27,021,00659456,27021,Cass,Cass County,06,H1,G4020,NaN,14660,None,A,5.235770e+09,1.016740e+09,+46.9514271,-094.3337731,1939-01-01,1930,NaT,1930,None,NaT,BORCHERT,2026-07-31,POINT (-10502637.934 5936460.776),2
2,bxq,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxq/y1939/bxq07189.jpg,bxq07189,46.966202,-94.389801,1939,3,3,85,0.0,85,27,021,00659456,27021,Cass,Cass County,06,H1,G4020,NaN,14660,None,A,5.235770e+09,1.016740e+09,+46.9514271,-094.3337731,1939-01-01,1930,NaT,1930,None,NaT,BORCHERT,2026-07-31,POINT (-10507424.583 5936559.09),3
3,bxq,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxq/y1939/bxq07191.jpg,bxq07191,46.966801,-94.432800,1939,4,4,85,0.0,85,27,021,00659456,27021,Cass,Cass County,06,H1,G4020,NaN,14660,None,A,5.235770e+09,1.016740e+09,+46.9514271,-094.3337731,1939-01-01,1930,NaT,1930,None,NaT,BORCHERT,2026-07-31,POINT (-10512211.244 5936656.784),4
4,bxq,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxq/y1939/bxq07195.jpg,bxq07195,46.967999,-94.518700,1939,5,5,85,0.0,85,27,021,00659456,27021,Cass,Cass County,06,H1,G4020,NaN,14660,None,A,5.235770e+09,1.016740e+09,+46.9514271,-094.3337731,1939-01-01,1930,NaT,1930,None,NaT,BORCHERT,2026-07-31,POINT (-10521773.51 5936852.176),5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115744,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-4A-041.jpg,WB-4A-041,43.948613,-92.515435,1940,0,0,0,0.0,0,NaN,NaN,NaN,NaN,Olmsted,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.000000e+00,0.000000e+00,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10298771.078 5457493.316),115745
115745,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-4A-043.jpg,WB-4A-043,43.948634,-92.558712,1940,0,0,0,0.0,0,NaN,NaN,NaN,NaN,Olmsted,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.000000e+00,0.000000e+00,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10303588.744 5457496.692),115746
115746,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-4A-045.jpg,WB-4A-045,43.948656,-92.601990,1940,0,0,0,0.0,0,NaN,NaN,NaN,NaN,Olmsted,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.000000e+00,0.000000e+00,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10308406.41 5457500.067),115747
115747,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-4A-047.jpg,WB-4A-047,43.948678,-92.645268,1940,0,0,0,0.0,0,NaN,NaN,NaN,NaN,Olmsted,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,0.000000e+00,0.000000e+00,NaN,NaN,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,POINT (-10313224.077 5457503.444),115748


### Adding filename identifier and validation

In [35]:
# add a new unique, human readable identifier field called "cty_year_id"
# it will be used for naming files
# it is a combination of the county, the year, and the photo ID. 

gdf_clean["cty_year_id"] = (
    gdf_clean["name10"].str.strip().str.lower().str.replace(" ", "_").str.replace('.', '', regex=False)
    + "_"
    + gdf_clean["year_"].astype("string")
    + "_"
    + gdf_clean["photo_id"].str.strip().str.lower().str.replace(" ", "_")
)

# add a number to any non-unique values produced by this
counts = gdf_clean.groupby("cty_year_id").cumcount()

gdf_clean["cty_year_id"] += counts.map(
    lambda n: f"_{n}" if n > 0 else ""
)

In [37]:
# validate that the periods were handled correctly
gdf_clean_filtered = gdf_clean.loc[gdf_clean["name10"] == "St. Louis"]

gdf_clean_filtered

,flight_cod,url_link,photo_id,lat,long_,year_,objectid_1,in_fid,near_fid,near_dist,objectid_2,statefp10,countyfp10,countyns10,geoid10,name10,namelsad10,lsad10,classfp10,mtfcc10,csafp10,cbsafp10,metdivfp10,funcstat10,aland10,awater10,intptlat10,intptlon10,year_conve,decade,decade_dat,decade_str,created_us,created_da,last_edite,last_edi_1,geometry,source_id,cty_year_id
41857,cir,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/cir/y1961/cir_002_121.jpg,cir_002_121,48.379299,-93.070099,1961,43073,43045,32,0.0,32,27,137,00662850,27137,St. Louis,St. Louis County,06,H1,G4020,NaN,20260,None,A,1.618069e+10,1.585336e+09,+47.5838524,-092.4636448,1961-01-01,1960,NaT,1960,None,NaT,BORCHERT,2026-07-31,POINT (-10360516.018 6170190.286),41858,st_louis_1961_cir_002_121
41858,cir,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/cir/y1961/cir_002_122.jpg,cir_002_122,48.392399,-93.070396,1961,43074,43046,32,0.0,32,27,137,00662850,27137,St. Louis,St. Louis County,06,H1,G4020,NaN,20260,None,A,1.618069e+10,1.585336e+09,+47.5838524,-092.4636448,1961-01-01,1960,NaT,1960,None,NaT,BORCHERT,2026-07-31,POINT (-10360549.135 6172386.065),41859,st_louis_1961_cir_002_122
41859,cir,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/cir/y1961/cir_002_123.jpg,cir_002_123,48.405800,-93.068802,1961,43075,43047,32,0.0,32,27,137,00662850,27137,St. Louis,St. Louis County,06,H1,G4020,NaN,20260,None,A,1.618069e+10,1.585336e+09,+47.5838524,-092.4636448,1961-01-01,1960,NaT,1960,None,NaT,BORCHERT,2026-07-31,POINT (-10360371.636 6174632.968),41860,st_louis_1961_cir_002_123
41860,cir,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/cir/y1961/cir_003_120.jpg,cir_003_120,48.375900,-93.033997,1961,43076,43048,32,0.0,32,27,137,00662850,27137,St. Louis,St. Louis County,06,H1,G4020,NaN,20260,None,A,1.618069e+10,1.585336e+09,+47.5838524,-092.4636448,1961-01-01,1960,NaT,1960,None,NaT,BORCHERT,2026-07-31,POINT (-10356497.128 6169620.649),41861,st_louis_1961_cir_003_120
41861,cir,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/cir/y1961/cir_003_121.jpg,cir_003_121,48.389000,-93.032997,1961,43077,43049,32,0.0,32,27,137,00662850,27137,St. Louis,St. Louis County,06,H1,G4020,NaN,20260,None,A,1.618069e+10,1.585336e+09,+47.5838524,-092.4636448,1961-01-01,1960,NaT,1960,None,NaT,BORCHERT,2026-07-31,POINT (-10356385.864 6171816.281),41862,st_louis_1961_cir_003_121
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108032,Lake,https://geo.lib.umn.edu/Lake/1939/CIS-6-029.jpg,CIS-6-029,46.968823,-91.808008,1939,109311,109311,32,0.0,32,27,137,00662850,27137,St. Louis,St. Louis County,06,H1,G4020,NaN,20260,None,A,1.618069e+10,1.585336e+09,+47.5838524,-092.4636448,1939-01-01,1930,NaT,1930,None,NaT,BORCHERT,2025-12-15,POINT (-10220020.667 5936986.689),108033,st_louis_1939_cis-6-029
108033,Lake,https://geo.lib.umn.edu/Lake/1939/CIS-6-030.jpg,CIS-6-030,46.956342,-91.807631,1939,109312,109312,32,0.0,32,27,137,00662850,27137,St. Louis,St. Louis County,06,H1,G4020,NaN,20260,None,A,1.618069e+10,1.585336e+09,+47.5838524,-092.4636448,1939-01-01,1930,NaT,1930,None,NaT,BORCHERT,2025-12-15,POINT (-10219978.708 5934950.935),108034,st_louis_1939_cis-6-030
108034,Lake,https://geo.lib.umn.edu/Lake/1939/CIS-6-031.jpg,CIS-6-031,46.943862,-91.807254,1939,109313,109313,32,0.0,32,27,137,00662850,27137,St. Louis,St. Louis County,06,H1,G4020,NaN,20260,None,A,1.618069e+10,1.585336e+09,+47.5838524,-092.4636448,1939-01-01,1930,NaT,1930,None,NaT,BORCHERT,2025-12-15,POINT (-10219936.749 5932915.656),108035,st_louis_1939_cis-6-031
108035,Lake,https://geo.lib.umn.edu/Lake/1939/CIS-6-032.jpg,CIS-6-032,46.931381,-91.806877,1939,109314,109314,32,0.0,32,27,137,00662850,27137,St. Louis,St. Louis County,06,H1,G4020,NaN,20260,None,A,1.618069e+10,1.585336e+09,+47.5838524,-092.4636448,1939-01-01,1930,NaT,1930,None,NaT,BORCHERT,2025

In [36]:
# validate that this new field is unique and contains no nulls
# should return 

print(f"All unique values:", gdf_clean['cty_year_id'].is_unique)
print(f"Has null values:", gdf_clean['cty_year_id'].isna().any())
print(f"Has spaces:", gdf_clean['cty_year_id'].str.contains(' ').any())

All unique values: True
Has null values: False
Has spaces: False


In [ ]:
# save
gdf_clean.to_file("data/historic_airphoto_index/umn_index_cleaned_allcolumns.gpkg")

In [79]:
# open cleaned table
clean_starting_index_path = "data/historic_airphoto_index/umn_index_original_utm_clean.gpkg"

gdf_clean = gpd.read_file(clean_starting_index_path)

In [80]:
gdf = gdf_clean.copy()

In [81]:
gdf

,flight_cod,url_link,photo_id,lat,long_,year_,objectid_1,in_fid,near_fid,near_dist,...,year_conve,decade,decade_dat,decade_str,created_us,created_da,last_edite,last_edi_1,county_year_photo_id,geometry
0,bxq,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxq/y1939/bxq07185.jpg,bxq07185,46.964901,-94.303902,1939,1,1,85,0.0,...,1939-01-01,1930,NaT,1930,None,NaT,BORCHERT,2026-07-31,cass_1939_bxq07185,POINT (400805.799 5202088.774)
1,bxq,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxq/y1939/bxq07187.jpg,bxq07187,46.965599,-94.346802,1939,2,2,85,0.0,...,1939-01-01,1930,NaT,1930,None,NaT,BORCHERT,2026-07-31,cass_1939_bxq07187,POINT (397543.55 5202221.539)
2,bxq,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxq/y1939/bxq07189.jpg,bxq07189,46.966202,-94.389801,1939,3,3,85,0.0,...,1939-01-01,1930,NaT,1930,None,NaT,BORCHERT,2026-07-31,cass_1939_bxq07189,POINT (394273.664 5202345.624)
3,bxq,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxq/y1939/bxq07191.jpg,bxq07191,46.966801,-94.432800,1939,4,4,85,0.0,...,1939-01-01,1930,NaT,1930,None,NaT,BORCHERT,2026-07-31,cass_1939_bxq07191,POINT (391003.84 5202471.082)
4,bxq,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxq/y1939/bxq07195.jpg,bxq07195,46.967999,-94.518700,1939,5,5,85,0.0,...,1939-01-01,1930,NaT,1930,None,NaT,BORCHERT,2026-07-31,cass_1939_bxq07195,POINT (384471.978 5202727.237)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115744,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-4A-041.jpg,WB-4A-041,43.948613,-92.515435,1940,0,0,0,0.0,...,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,olmsted_1940_wb-4a-041,POINT (538883.127 4866279.502)
115745,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-4A-043.jpg,WB-4A-043,43.948634,-92.558712,1940,0,0,0,0.0,...,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,olmsted_1940_wb-4a-043,POINT (535410.355 4866262.453)
115746,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-4A-045.jpg,WB-4A-045,43.948656,-92.601990,1940,0,0,0,0.0,...,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,olmsted_1940_wb-4a-045,POINT (531937.586 4866247.225)
115747,Olmsted,https://geo.lib.umn.edu/Olmsted/1940/WB-4A-047.jpg,WB-4A-047,43.948678,-92.645268,1940,0,0,0,0.0,...,NaT,1940,NaT,1940,None,NaT,BORCHERT,2025-12-15,olmsted_1940_wb-4a-047,POINT (528464.82 4866233.818)


## Next Steps
- Drop some columns
- rename some columns
- add some fields

### Adding fields

columns to add: 

- county_year_photo
- download_filename
- output_filename
- processed (boolean)
- scale
- 
- ml_confidence
- gcp_count
- error_log

#### filenames

In [16]:
# create a safe file name column
gdf["download_filename"] = (
    gdf["filename"]
    .astype(str)
    .str.lstrip("/")
    .str.replace("/", "_")
)

In [17]:
# create a safe output filename column
gdf["output_filename"] = (
    gdf["download_filename"].apply(lambda f: f.replace(".jpg", "_georef.tif"))
)

In [18]:
gdf["processed"] = False

In [19]:
gdf["scale"] = None

In [22]:
gdf

,OBJECTID_1,filename,provider,project,year,photo_id,ext,xutm,yutm,status,image_url,globalid,geometry,download_filename,output_filename,processed,scale
0,17660,/usda/bjf/y1938/bjf02002.jpg,usda,bjf,1938,bjf02002,jpg,260432.0,4868321.0,1,https://maps.dnr.state.mn.us/airphotos/usda/bjf/y1938/bjf02002.jpg,{6C21141B-1856-46E2-B199-FB9678486151},POINT (260431.708 4868321.904),usda_bjf_y1938_bjf02002.jpg,usda_bjf_y1938_bjf02002_georef.tif,False,None
1,17662,/usda/bjf/y1938/bjf02004.jpg,usda,bjf,1938,bjf02004,jpg,257333.0,4868367.0,1,https://maps.dnr.state.mn.us/airphotos/usda/bjf/y1938/bjf02004.jpg,{967F845D-5444-40B6-BF05-91437D1CAF63},POINT (257332.011 4868368.265),usda_bjf_y1938_bjf02004.jpg,usda_bjf_y1938_bjf02004_georef.tif,False,None
2,17664,/usda/bjf/y1938/bjf02006.jpg,usda,bjf,1938,bjf02006,jpg,254233.0,4868414.0,1,https://maps.dnr.state.mn.us/airphotos/usda/bjf/y1938/bjf02006.jpg,{04683AF5-AE59-4778-AC4B-849119DEC0B0},POINT (254232.313 4868414.627),usda_bjf_y1938_bjf02006.jpg,usda_bjf_y1938_bjf02006_georef.tif,False,None
3,17665,/usda/bjf/y1938/bjf02012.jpg,usda,bjf,1938,bjf02012,jpg,253965.0,4865651.0,1,https://maps.dnr.state.mn.us/airphotos/usda/bjf/y1938/bjf02012.jpg,{8245AC47-1BED-42D5-81E2-79CDEF7E44BF},POINT (253964.528 4865651.395),usda_bjf_y1938_bjf02012.jpg,usda_bjf_y1938_bjf02012_georef.tif,False,None
4,17667,/usda/bjf/y1938/bjf02014.jpg,usda,bjf,1938,bjf02014,jpg,256931.0,4865569.0,1,https://maps.dnr.state.mn.us/airphotos/usda/bjf/y1938/bjf02014.jpg,{4F0BDE31-A7AA-4781-A669-29CF8E7C3DE2},POINT (256930.424 4865569.669),usda_bjf_y1938_bjf02014.jpg,usda_bjf_y1938_bjf02014_georef.tif,False,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93756,96459,/usda/byg/y1966/byg05gg117.jpg,usda,byg,1966,byg05gg117,jpg,319917.0,5429225.0,1,https://maps.dnr.state.mn.us/airphotos/usda/byg/y1966/byg05gg117.jpg,{FFE59653-BF0D-4BFE-AA97-F5B512DEAA4B},POINT (319916.86 5429226.027),usda_byg_y1966_byg05gg117.jpg,usda_byg_y1966_byg05gg117_georef.tif,False,None
93757,96460,/usda/byg/y1966/byg05gg119.jpg,usda,byg,1966,byg05gg119,jpg,323196.0,5429165.0,1,https://maps.dnr.state.mn.us/airphotos/usda/byg/y1966/byg05gg119.jpg,{FB5FE609-92DD-4B07-AC46-624C3DB7959D},POINT (323195.16 5429166.422),usda_byg_y1966_byg05gg119.jpg,usda_byg_y1966_byg05gg119_georef.tif,False,None
93758,96461,/usda/byg/y1966/byg05gg121.jpg,usda,byg,1966,byg05gg121,jpg,326395.0,5429046.0,1,https://maps.dnr.state.mn.us/airphotos/usda/byg/y1966/byg05gg121.jpg,{2933DB1E-682F-464A-99F8-369190014377},POINT (326393.986 5429047.211),usda_byg_y1966_byg05gg121.jpg,usda_byg_y1966_byg05gg121_georef.tif,False,None
93759,96462,/usda/byg/y1966/byg05gg123.jpg,usda,byg,1966,byg05gg123,jpg,329683.0,5428977.0,1,https://maps.dnr.state.mn.us/airphotos/usda/byg/y1966/byg05gg123.jpg,{7E07CF60-6D46-4F27-8107-A47602220FB5},POINT (329682.221 5428977.671),usda_byg_y1966_byg05gg123.jpg,usda_byg_y1966_byg05gg123_georef.tif,False,None


**Important: photo_id is NOT unique, but filename and download_filename are**

In [20]:
print(gdf['photo_id'].is_unique)  

print(gdf['filename'].is_unique)  

print(gdf['download_filename'].is_unique)

False
True
True


### column sorting

In [23]:
# inspect
gdf.columns

Index(['OBJECTID_1', 'filename', 'provider', 'project', 'year', 'photo_id',
       'ext', 'xutm', 'yutm', 'status', 'image_url', 'globalid', 'geometry',
       'download_filename', 'output_filename', 'processed', 'scale'],
      dtype='str')

In [24]:
gdf_columnorder = gdf[['filename', 'project', 'year', 'scale', 'geometry', 'provider', 'image_url', 
                       'download_filename', 'output_filename', 'OBJECTID_1', 'photo_id', 'status', 'xutm', 'yutm', 'ext', 'globalid']]

gdf_columnorder

,filename,project,year,scale,geometry,provider,image_url,download_filename,output_filename,OBJECTID_1,photo_id,status,xutm,yutm,ext,globalid
0,/usda/bjf/y1938/bjf02002.jpg,bjf,1938,None,POINT (260431.708 4868321.904),usda,https://maps.dnr.state.mn.us/airphotos/usda/bjf/y1938/bjf02002.jpg,usda_bjf_y1938_bjf02002.jpg,usda_bjf_y1938_bjf02002_georef.tif,17660,bjf02002,1,260432.0,4868321.0,jpg,{6C21141B-1856-46E2-B199-FB9678486151}
1,/usda/bjf/y1938/bjf02004.jpg,bjf,1938,None,POINT (257332.011 4868368.265),usda,https://maps.dnr.state.mn.us/airphotos/usda/bjf/y1938/bjf02004.jpg,usda_bjf_y1938_bjf02004.jpg,usda_bjf_y1938_bjf02004_georef.tif,17662,bjf02004,1,257333.0,4868367.0,jpg,{967F845D-5444-40B6-BF05-91437D1CAF63}
2,/usda/bjf/y1938/bjf02006.jpg,bjf,1938,None,POINT (254232.313 4868414.627),usda,https://maps.dnr.state.mn.us/airphotos/usda/bjf/y1938/bjf02006.jpg,usda_bjf_y1938_bjf02006.jpg,usda_bjf_y1938_bjf02006_georef.tif,17664,bjf02006,1,254233.0,4868414.0,jpg,{04683AF5-AE59-4778-AC4B-849119DEC0B0}
3,/usda/bjf/y1938/bjf02012.jpg,bjf,1938,None,POINT (253964.528 4865651.395),usda,https://maps.dnr.state.mn.us/airphotos/usda/bjf/y1938/bjf02012.jpg,usda_bjf_y1938_bjf02012.jpg,usda_bjf_y1938_bjf02012_georef.tif,17665,bjf02012,1,253965.0,4865651.0,jpg,{8245AC47-1BED-42D5-81E2-79CDEF7E44BF}
4,/usda/bjf/y1938/bjf02014.jpg,bjf,1938,None,POINT (256930.424 4865569.669),usda,https://maps.dnr.state.mn.us/airphotos/usda/bjf/y1938/bjf02014.jpg,usda_bjf_y1938_bjf02014.jpg,usda_bjf_y1938_bjf02014_georef.tif,17667,bjf02014,1,256931.0,4865569.0,jpg,{4F0BDE31-A7AA-4781-A669-29CF8E7C3DE2}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93756,/usda/byg/y1966/byg05gg117.jpg,byg,1966,None,POINT (319916.86 5429226.027),usda,https://maps.dnr.state.mn.us/airphotos/usda/byg/y1966/byg05gg117.jpg,usda_byg_y1966_byg05gg117.jpg,usda_byg_y1966_byg05gg117_georef.tif,96459,byg05gg117,1,319917.0,5429225.0,jpg,{FFE59653-BF0D-4BFE-AA97-F5B512DEAA4B}
93757,/usda/byg/y1966/byg05gg119.jpg,byg,1966,None,POINT (323195.16 5429166.422),usda,https://maps.dnr.state.mn.us/airphotos/usda/byg/y1966/byg05gg119.jpg,usda_byg_y1966_byg05gg119.jpg,usda_byg_y1966_byg05gg119_georef.tif,96460,byg05gg119,1,323196.0,5429165.0,jpg,{FB5FE609-92DD-4B07-AC46-624C3DB7959D}
93758,/usda/byg/y1966/byg05gg121.jpg,byg,1966,None,POINT (326393.986 5429047.211),usda,https://maps.dnr.state.mn.us/airphotos/usda/byg/y1966/byg05gg121.jpg,usda_byg_y1966_byg05gg121.jpg,usda_byg_y1966_byg05gg121_georef.tif,96461,byg05gg121,1,326395.0,5429046.0,jpg,{2933DB1E-682F-464A-99F8-369190014377}
93759,/usda/byg/y1966/byg05gg123.jpg,byg,1966,None,POINT (329682.221 5428977.671),usda,https://maps.dnr.state.mn.us/airphotos/usda/byg/y1966/byg05gg123.jpg,usda_byg_y1966_byg05gg123.jpg,usda_byg_y1966_byg05gg123_georef.tif,96462,byg05gg123,1,329683.0,5428977.0,jpg,{7E07CF60-6D46-4F27-8107-A47602220FB5}


In [ ]:
# save that 

gdf = gdf_columnorder

print(gdf.columns)

Index(['filename', 'project', 'year', 'scale', 'geometry', 'provider',
       'image_url', 'download_filename', 'output_filename', 'OBJECTID_1',
       'photo_id', 'status', 'xutm', 'yutm', 'ext', 'globalid'],
      dtype='str')


### Sorting

In [27]:
gdf_sorted = gdf.sort_values(by=['project','year'], ascending=[True, True])

gdf_sorted

,filename,project,year,scale,geometry,provider,image_url,download_filename,output_filename,OBJECTID_1,photo_id,status,xutm,yutm,ext,globalid
25182,/projects/ait/y1975/ait_7_162.jpg,ait,1975,None,POINT (470069.123 5108393.671),projects,https://maps.dnr.state.mn.us/airphotos/projects/ait/y1975/ait_7_162.jpg,projects_ait_y1975_ait_7_162.jpg,projects_ait_y1975_ait_7_162_georef.tif,154847,ait_7_162,1,470070.0,5108393.0,jpg,{CE373997-6112-49D2-9102-72273DF606B2}
25183,/projects/ait/y1975/ait_7_163.jpg,ait,1975,None,POINT (470165.516 5109793.468),projects,https://maps.dnr.state.mn.us/airphotos/projects/ait/y1975/ait_7_163.jpg,projects_ait_y1975_ait_7_163.jpg,projects_ait_y1975_ait_7_163_georef.tif,154848,ait_7_163,1,470166.0,5109793.0,jpg,{4B76F451-19FF-4185-BA18-7DF2B1FEE4FE}
25184,/projects/ait/y1975/ait_7_164.jpg,ait,1975,None,POINT (470148.752 5111109.445),projects,https://maps.dnr.state.mn.us/airphotos/projects/ait/y1975/ait_7_164.jpg,projects_ait_y1975_ait_7_164.jpg,projects_ait_y1975_ait_7_164_georef.tif,154849,ait_7_164,1,470149.0,5111109.0,jpg,{0C3B9C34-ABE3-4E53-8D13-F2C86CBA6487}
25185,/projects/ait/y1975/ait_7_165.jpg,ait,1975,None,POINT (470219.999 5112291.309),projects,https://maps.dnr.state.mn.us/airphotos/projects/ait/y1975/ait_7_165.jpg,projects_ait_y1975_ait_7_165.jpg,projects_ait_y1975_ait_7_165_georef.tif,154850,ait_7_165,1,470221.0,5112290.0,jpg,{86F4FF7A-77BD-4C2D-B925-93CDBEC4F5B0}
25186,/projects/ait/y1975/ait_7_166.jpg,ait,1975,None,POINT (470266.1 5113456.409),projects,https://maps.dnr.state.mn.us/airphotos/projects/ait/y1975/ait_7_166.jpg,projects_ait_y1975_ait_7_166.jpg,projects_ait_y1975_ait_7_166_georef.tif,154851,ait_7_166,1,470267.0,5113456.0,jpg,{D0E3C8A8-0D65-4B99-B83E-F08008B6764F}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16085,/projects/wp/y1963/wp_3dd_120.jpg,wp,1963,None,POINT (406050.215 5005518.257),projects,https://maps.dnr.state.mn.us/airphotos/projects/wp/y1963/wp_3dd_120.jpg,projects_wp_y1963_wp_3dd_120.jpg,projects_wp_y1963_wp_3dd_120_georef.tif,165097,wp_3dd_120,1,406051.0,5005517.0,jpg,{ED680899-C440-443F-9EA8-5EE22C99FF9E}
16086,/projects/wp/y1963/wp_3dd_158.jpg,wp,1963,None,POINT (401159.582 5018139.933),projects,https://maps.dnr.state.mn.us/airphotos/projects/wp/y1963/wp_3dd_158.jpg,projects_wp_y1963_wp_3dd_158.jpg,projects_wp_y1963_wp_3dd_158_georef.tif,165127,wp_3dd_158,1,401160.0,5018139.0,jpg,{562E7926-0A26-4A6C-8215-40C0E844C537}
16087,/projects/wp/y1963/wp_3dd_159.jpg,wp,1963,None,POINT (402868.1 5018054.507),projects,https://maps.dnr.state.mn.us/airphotos/projects/wp/y1963/wp_3dd_159.jpg,projects_wp_y1963_wp_3dd_159.jpg,projects_wp_y1963_wp_3dd_159_georef.tif,165128,wp_3dd_159,1,402869.0,5018054.0,jpg,{8651A8B0-984A-4553-A1A1-E370031940AB}
16088,/projects/wp/y1963/wp_3dd_160.jpg,wp,1963,None,POINT (404149.488 5018097.22),projects,https://maps.dnr.state.mn.us/airphotos/projects/wp/y1963/wp_3dd_160.jpg,projects_wp_y1963_wp_3dd_160.jpg,projects_wp_y1963_wp_3dd_160_georef.tif,165129,wp_3dd_160,1,404150.0,5018096.0,jpg,{C7CBE4EF-139A-4190-9532-D3EE8A9F8C45}


In [28]:
# save that

gdf = gdf_sorted

### Export

In [29]:
gdf.to_file("data/historic_airphoto_index/main_airphoto_index.gpkg")